In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import argparse
from collections import defaultdict
import math
import sys

In [ ]:
df1 = pd.read_csv('Haitian_Foods_Story.csv', header=None)
df2 = pd.read_csv('French_Foods_Story.csv',header=None)
df0 = pd.concat([df1, df2], ignore_index=True)

In [ ]:
df1

,0
0,**Diri Kole ak Pwa nan Lakou Granmè Zilè**\n\n...
1,**Diri blan ak sòs pwa nan lakou Granmè Zil**\...
2,**Diri blan ak Sòs pwa nwa**\n\nTi Granmè Zilm...
3,**Diri blan ak Sòs Pwa Blan**\n\nTi Mari se yo...
4,**Diri Djondjon pou Ti Granmè**\n\nTi Granmè t...
...,...
177,**Mabi a Granmè Zilè**\n\nGranmè Zilè te gen y...
178,**Godrin Mabi ak Ananas Magik**\n\nTi Bouki te...
179,**Ti Istwa Punch**\n\nLè solèy la te kouche so...
180,**Jimex nan Mitan Nuit**\n\nTi Kako te gen yon...


In [ ]:
def get_text_column(df):
    if isinstance(df, pd.Series):
        return df
    else:
        return df.iloc[:, 0]

df1 = get_text_column(df1)
df2 = get_text_column(df2)
df0 = get_text_column(df0)

In [ ]:
STOPWORDS_HT = set([
    "mwen", "ou", "li", "nou", "yo",
    "sa", "ki", "kisa",
    "nan", "sou", "ak", "pou", "pa",
    "se", "ye", "te", "ap", "pral",
    "gen", "fè", "di",
    "la", "a", "an", "lan",
    "yon", "youn",
    "men", "oswa", "paske",
    "kòm", "tankou",
    "isit", "la", "là",
    "tout", "plis", "mwens",
    "byen", "mal", "m", "ka","l", " ","t","w",
    "vin","vini","k","san","manje","moun", "si","ale","tt","sak","pi",""
])

In [ ]:
def get_log_odds(col1, col2, col0,verbose=False,lower=True):
    """Monroe et al. Fightin' Words method to identify top words in df1 and df2
    against df0 as the background corpus"""
    if lower:
        counts1 = defaultdict(int, [[i,j] for i,j in col1.str.lower().str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        counts2 = defaultdict(int,[[i,j] for i,j in col2.str.lower().str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        prior = defaultdict(int,[[i,j] for i,j in col0.str.lower().str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
    else:
        counts1 = defaultdict(int,[[i,j] for i,j in col1.str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        counts2 = defaultdict(int,[[i,j] for i,j in col2.str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])
        prior = defaultdict(int,[[i,j] for i,j in col0.str.split(expand=True).stack().replace(r'[^a-zA-Z\s]', '', regex=True).value_counts().items()])

    sigmasquared = defaultdict(float)
    sigma = defaultdict(float)
    delta = defaultdict(float)

    for word in prior.keys():
        prior[word] = int(prior[word] + 0.5)

    for word in counts2.keys():
        counts1[word] = int(counts1[word] + 0.5)
        if prior[word] == 0:
            prior[word] = 1

    for word in counts1.keys():
        counts2[word] = int(counts2[word] + 0.5)
        if prior[word] == 0:
            prior[word] = 1

    n1 = sum(counts1.values())
    n2 = sum(counts2.values())
    nprior = sum(prior.values())

    for word in prior.keys():
        if prior[word] > 0:
            l1 = float(counts1[word] + prior[word]) / (( n1 + nprior ) - (counts1[word] + prior[word]))
            l2 = float(counts2[word] + prior[word]) / (( n2 + nprior ) - (counts2[word] + prior[word]))
            sigmasquared[word] =  1/(float(counts1[word]) + float(prior[word])) + 1/(float(counts2[word]) + float(prior[word]))
            sigma[word] =  math.sqrt(sigmasquared[word])
            delta[word] = ( math.log(l1) - math.log(l2) ) / sigma[word]

    if verbose:
        for word in sorted(delta, key=delta.get)[:10]:
            print("%s, %.3f" % (word, delta[word]))

        for word in sorted(delta, key=delta.get,reverse=True)[:10]:
            print("%s, %.3f" % (word, delta[word]))
    return delta

In [ ]:
food_results = get_log_odds(get_text_column(df1),get_text_column(df2),get_text_column(df0),verbose=False,lower=True)

In [ ]:
haitian_food_dict = defaultdict(float)
french_food_dict = defaultdict(float)
for word , score  in food_results.items():
  if score > 1.96 and word not in STOPWORDS_HT:
    haitian_food_dict[word] = food_results[word]
  if score <- 1.96  and word not in STOPWORDS_HT:
    french_food_dict[word] = food_results[word]

In [ ]:
haitian_food_dict= dict(sorted(haitian_food_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
haitian_food_dict

{'ss': 6.287928803288579,
 'pwason': 6.208536010847571,
 'mayi': 6.178437292286819,
 'zil': 6.015282733243587,
 'klm': 5.622323127284787,
 'diri': 5.267867101057629,
 'arans': 5.110077186122919,
 'pitimi': 4.9120783725935455,
 'granm': 4.751808517715215,
 'bannann': 4.319716836482945,
 'bf': 4.247350224585178,
 'pate': 4.238215467785052,
 'janik': 4.185123423631397,
 'ragou': 4.064904152015945,
 'lam': 3.8438880390306225,
 'manyk': 3.769535202819181,
 'tablt': 3.769535202819181,
 'labouyi': 3.725431178558025,
 'griyo': 3.6598452458852053,
 'taso': 3.6356204991837413,
 'jimex': 3.4965873505427125,
 'klmns': 3.4965873505427125,
 'ji': 3.4965873505427125,
 'kabrit': 3.468697948791899,
 'tibo': 3.3606398211696495,
 'bouyon': 3.3564729819385435,
 'pwa': 3.279260149864997,
 'kaka': 3.2535490266798845,
 'espageti': 3.2516974450966107,
 'e': 3.242855439880214,
 'malta': 3.200474060701258,
 'patat': 3.1690580816989535,
 'by': 3.1484182144493076,
 'ze': 3.113088812981295,
 'ble': 3.0984341087505

In [ ]:
french_food_dict = dict(sorted(french_food_dict.items(), key=lambda item: item[1]))

In [ ]:
french_food_dict

{'marilne': -6.4715999373153235,
 'fromaj': -6.4559864014383095,
 'lili': -6.111390145919218,
 'tante': -5.783708005372103,
 'soupe': -5.368081637221219,
 'pain': -4.984849042862058,
 'tarte': -4.858554930242061,
 'manmi': -4.673487838921151,
 'mariln': -4.5616573043308355,
 'lizt': -4.184522387154441,
 'plat': -4.097079797408957,
 'epui': -4.01930987013629,
 'moso': -4.013896118329931,
 'zaza': -3.860771222155983,
 'mz': -3.501748787903641,
 'ufs': -3.479931435965474,
 'restoran': -3.458147917630464,
 'gat': -3.38950202797054,
 'tijo': -3.3786778626792096,
 'maris': -3.3331150426959626,
 'vani': -3.287632484649155,
 'monsy': -3.258253815740643,
 'graten': -3.249147842947387,
 'mireille': -3.2009973211764455,
 'bouch': -3.048393262407174,
 'fonn': -3.02991683068681,
 'boudin': -3.0006866389812306,
 'le': -3.0006866389812306,
 'madanm': -2.9568915466649752,
 'souvenans': -2.9267355667798576,
 'lodie': -2.8412097623964567,
 'fmaj': -2.8412097623964567,
 'baba': -2.8412097623964567,
 'fme

Artist

In [ ]:
df_haitian = pd.read_csv('Haitian_Musical_Groups_Story.csv', header=None)
df_french = pd.read_csv('French_Artists_Story.csv',header=None)
df_all = pd.concat([df_haitian, df_french], ignore_index=True)

In [ ]:
df_haitian

,0
0,**Lè Mizik Tabou Combo Te Fè Kè Jean-Pierre Ba...
1,**Lè Mizik T-Vice Te Fè Kè Roberto Bat Tanbou ...
2,**Ti Istwa a**\n\n*Lè sol la te kouche sou Por...
3,"**Ti Istwa a : ""Mizik Nu-look nan Kè Mwen""**\n..."
4,**Lè Mizik Harmonik a Te Fè Kè Li Chante**\n\n...
...,...
193,**Ti Mizik ki Te Fè Kè Li Klere**\n\nLè solèy ...
194,**Lè Gazzman Te Fè Kè Li Pale**\n\nTi Gwo Wozo...
195,"**Ti Istwa a : ""Mizik ki te fè lè l vin nwa""**..."
196,**Lè Mizik an Te Fè Lavi Li Klere**\n\nTi Klod...


In [ ]:
artist_results = get_log_odds(get_text_column(df_haitian),get_text_column(df_french),get_text_column(df_all),verbose=False,lower=True)

In [ ]:
haitian_artist_dict = {}
french_artist_dict = {}
for word , score  in artist_results.items():
  if score > 1.96 and word not in STOPWORDS_HT:
    haitian_artist_dict[word] = artist_results[word]
  if score <- 1.96 and word not in STOPWORDS_HT:
    french_artist_dict[word] = artist_results[word]

In [ ]:
haitian_artist_dict = dict(sorted(haitian_artist_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
haitian_artist_dict

{'bouki': 4.691559581908997,
 'mesye': 4.038493057866309,
 'kako': 3.897993610661615,
 'tanbou': 3.8481557064471605,
 'rozo': 3.784550535734445,
 'gwo': 3.688296334469279,
 'kaka': 3.601756165501541,
 'jilo': 3.4420534335105244,
 'obas': 3.4132035204417797,
 'gabel': 3.2179511238144474,
 'bwa': 3.2098812646453005,
 'yole': 3.11574453274719,
 'jj': 3.06560511257279,
 'man': 3.0556788608364047,
 'manti': 2.982581819931784,
 'jo': 2.9039351450977167,
 'wozo': 2.8980686784319234,
 'josu': 2.708088864928788,
 'gita': 2.6403653872750956,
 'zenglen': 2.6067313900188975,
 'alex': 2.5438998646719835,
 'bat': 2.523165361804259,
 'ginen': 2.481815149357855,
 'tijack': 2.479477632808562,
 'emisyon': 2.4357832897329637,
 'mak': 2.413337309616308,
 'richardo': 2.413337309616308,
 'toby': 2.413337309616308,
 'jakito': 2.413337309616308,
 'durosier': 2.413337309616308,
 'chante': 2.364543852997718,
 'jc': 2.3453335448415986,
 'gwozo': 2.3453335448415986,
 'drose': 2.3453335448415986,
 'guy': 2.3453335

In [ ]:
french_artist_dict = dict(sorted(french_artist_dict.items(), key=lambda item: item[1]))

In [ ]:
french_artist_dict

{'jasmin': -5.028735442922997,
 'jai': -4.3990593460945195,
 'le': -4.312659636486192,
 'frans': -4.138857237501735,
 'jude': -4.046745621488514,
 'les': -4.045380098887226,
 'dany': -3.9020647180284507,
 'et': -3.778115651193731,
 'jeanmax': -3.6499656542623207,
 'klmn': -3.5171555655075513,
 'paris': -3.4257597107486584,
 'mireille': -3.4187121153494373,
 'des': -3.3681348125562214,
 'on': -3.197738766468933,
 'franse': -3.1858324207260678,
 'un': -3.084645973391216,
 'portauprince': -3.0655441784788895,
 'suis': -3.0327881829050445,
 'rap': -3.0307236332240337,
 'djo': -2.9108259538752623,
 'manouchka': -2.89671421242841,
 'me': -2.8672251247718203,
 'pas': -2.85398463718101,
 'radio': -2.8470155704765445,
 'jeanphilippe': -2.815831138286122,
 'lara': -2.758929509105151,
 'une': -2.758929509105151,
 'que': -2.758929509105151,
 'diski': -2.758929509105151,
 'tu': -2.758929509105151,
 'lodie': -2.758929509105151,
 'tijol': -2.700830206430633,
 'mzilz': -2.700830206430633,
 'mwa': -2.6

In [ ]:
df_haitian_person = pd.read_csv('Haitian_people_names_stories.csv', header=None)
df_french_person = pd.read_csv('French_people_name_stories.csv',header=None)
df_all_person = pd.concat([df_haitian_person, df_french_person], ignore_index=True)

In [ ]:
df_haitian_person

,0
0,"**Colette ak Chèz la**\n\nPort-au-Prince, nan ..."
1,**Wilson ak Chwal Bwa**\n\nWilson te gen yon t...
2,**Albert ak Bwa Mango**\n\nAlbert te gen yon t...
3,**Anthony ak Bwa Mango**\n\nAnthony te gen yon...
4,**Anderson ak Chèz la**\n\nAnderson te gen yon...
...,...
145,**Magloire ak Chwal Bò Kòd**\n\nMagloire te ge...
146,**Maxime ak Chèz la nan Lavil**\n\nMaxime te g...
147,**Jean-Baptiste ak Chwal Bòkò**\n\nJean-Baptis...
148,**Tamzin ak Chèz la nan Vilaj La**\n\nVilaj La...


In [ ]:
person_results = get_log_odds(get_text_column(df_haitian_person),get_text_column(df_french_person),get_text_column(df_all_person),verbose=False,lower=True)

In [ ]:
haitian_person_dict = defaultdict(float)
french_person_dict = defaultdict(float)
for word , score  in person_results.items():
  if score > 1.96 and word not in STOPWORDS_HT:
    haitian_person_dict[word] = person_results[word]
  if score < -1.96  and word not in STOPWORDS_HT:
    french_person_dict[word] = person_results[word]

In [ ]:
haitian_person_dict = dict(sorted(haitian_person_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
haitian_person_dict

{'chwal': 8.395173912615796,
 'bwa': 5.921806523837485,
 'mango': 5.908805724011857,
 'tap': 5.436088872174385,
 '': 5.156656154078175,
 'dlo': 4.919106731980272,
 'granm': 4.053057354424284,
 'lajan': 3.9698820691306587,
 'bt': 3.9243762074770054,
 'machann': 3.642304902463652,
 'mang': 3.4916802903747324,
 'mn': 3.4035693008332695,
 'kd': 3.2652090626944346,
 'lakay': 3.2644922127061706,
 'kouri': 3.1888486586233533,
 'koupe': 3.123983587031235,
 'ank': 3.070488277481679,
 'zn': 2.998323858956178,
 'arnold': 2.9600001274555794,
 'monte': 2.9196621307912265,
 'edris': 2.906649945939278,
 'monsy': 2.8578481337839112,
 'eloi': 2.8523030055914527,
 'vann': 2.8323812363917167,
 'msye': 2.775419808113132,
 'dadi': 2.740380559628676,
 'zwazo': 2.7403198699315383,
 'va': 2.7170041897020374,
 'mapou': 2.7122962744400216,
 'pran': 2.6918088556169066,
 'nwit': 2.6910409943603457,
 'gason': 2.6875222020451313,
 'wilson': 2.682670359881246,
 'jude': 2.682670359881246,
 'magloire': 2.6826703598812

In [ ]:
french_person_dict = dict(sorted(french_person_dict.items(), key=lambda item: item[1]))

In [ ]:
french_person_dict

{'frans': -8.908483665856023,
 'vilaj': -5.541926711398781,
 'mari': -5.441499248510679,
 'roche': -5.373349124682374,
 'claire': -5.012796441086835,
 'kle': -4.9351112253852945,
 'lodie': -4.925931605193525,
 'paris': -4.473199292784024,
 'vizaj': -4.074768096714514,
 'marie': -3.7144535000892422,
 'ekri': -3.6517262235708348,
 'gri': -3.6446161570596187,
 'liv': -3.644268006624797,
 'peyi': -3.5695490152089655,
 'aprann': -3.5530628629507914,
 'tren': -3.4834485758703995,
 'kreyl': -3.4304969440923205,
 'marcel': -3.3879643490603075,
 'kafe': -3.330456722994838,
 'restoran': -3.299488829701659,
 'renmen': -3.292746518711081,
 'pandan': -3.2590526835988336,
 'fi': -3.219216845735961,
 'valiz': -3.164957993980338,
 'foto': -3.1562861492661685,
 'dubois': -3.0871355439033694,
 'hlose': -3.083845502645026,
 'kote': -3.0769685903092485,
 'madame': -3.066625392850505,
 'lyon': -3.00824372836839,
 'zy': -2.9868400283468786,
 'lavi': -2.9652777540150845,
 'souri': -2.926023201544804,
 'claud